<a href="https://www.kaggle.com/code/buianhtruc/comfyui-in-kaggle?scriptVersionId=334889223" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

remember: GPU T4 X2 and on Internet in Session options

Create CLOUDFLARE_TUNNEL_TOKEN or (NGROK_AUTHTOKEN + NGROK_DOMAIN) if you want to use a fixed domain (Add-ons -> Secrets -> Add a new secret)

docs: https://docs.comfy.org/

## Optional: Custom Google API Client ID/Secret (avoid rclone rate limits)

rclone by default uses shared OAuth app -> may hit Google rate limits (429 Too Many Requests) when many users online.

To use YOUR OWN Google API credentials:

1. Go to https://console.cloud.google.com/
2. Create new project (any name, e.g. "rclone-backup")
3. APIs & Services -> Library -> search "Google Drive API" -> Enable
4. APIs & Services -> OAuth consent screen:
   - User type: External -> Create
   - App name: anything (e.g. "My Rclone Backup")
   - User support email: your email
   - Developer contact: your email
   - Save and Continue through scopes (no scopes needed here, rclone will request them in authorize)
   - Add yourself to "Test users" section (your email)
   - Save (app stays in "Testing" status which is fine for personal use)
5. APIs & Services -> Credentials:
   - Create Credentials -> OAuth client ID
   - Application type: **Desktop app** (important: not Web app)
   - Name: anything
   - Create -> copy **Client ID** + **Client Secret**
6. Back in Kaggle notebook: Add-ons -> Secrets -> Add new secret (2 secrets):
   - Label: `GDRIVE_CLIENT_ID`        Value: your Client ID
   - Label: `GDRIVE_CLIENT_SECRET`    Value: your Client Secret
7. Cell 2 prompt "Use custom Google API Client ID/Secret? (y/N)" -> answer "y"

## Optional: HuggingFace / CivitAI tokens (for model downloads)

Some models on HuggingFace are gated (require login) or on CivitAI (require API key).
If you need to download such models:

- **HuggingFace**: https://huggingface.co/settings/tokens -> create token -> add to Kaggle Secrets:
  - Label: `HF_TOKEN`        Value: your HuggingFace token (e.g. `hf_...`)
- **CivitAI**: https://civitai.com/user/account -> copy API Key -> add to Kaggle Secrets:
  - Label: `CIVITAI_API_KEY` Value: your CivitAI API key

Notes:
- First time switching from rclone default to custom credentials: existing `RCLONE_DRIVE_TOKEN` will be auto-detected as stale and re-auth will be triggered. Just follow the OAuth link once more.
- App in "Testing" status requires your Google account to be in "Test users" list. Publish is optional.
- Custom credentials have their own Google Drive API quota -> avoids shared rclone rate limits.

# Setup

In [ ]:
# ================= GOOGLE DRIVE CONFIG =================
REMOTE = "gdrive"
SRC = "/root/comfy/ComfyUI"
DRIVE_BACKUP_DIR = "ComfyUI"

INCLUDES = [
    "blueprints",
    "input",
    "user",
    "output",
]

EXCLUDES = [
    "input/**/__pycache__/**",
    "output/**/__pycache__/**",
    "**/*.tmp",
    "**/*.log",
]

SYNC_INTERVAL = 300  # seconds between rclone sync

# ================= SETUP CHOICES =================
drive = input("Configure Google Drive backup? (y/N): ").strip().lower()
CONFIGURE_DRIVE = drive == "y"


# ================= CUSTOM GOOGLE API (optional) =================
use_custom = input("Use custom Google API Client ID/Secret? (y/N): ").strip().lower()
USE_CUSTOM_GDRIVE_API = use_custom == "y"
if USE_CUSTOM_GDRIVE_API:
    print("  -> Using custom Client ID/Secret (from Kaggle Secrets GDRIVE_CLIENT_ID/SECRET)")
    print("  -> Required: see setup guide in Cell 0 above")
else:
    print("  -> Using rclone default OAuth (may hit rate limits on busy days)")


print("Tunnel mode:")
print("  1) Quick Tunnel (Cloudflare - khong can cau hinh)")
print("  2) Named Tunnel (can CLOUDFLARE_TUNNEL_TOKEN)")
print("  3) Ngrok (can NGROK_AUTHTOKEN + NGROK_DOMAIN)")
choice = input("Chon (1/2/3): ").strip()
TUNNEL_MODE = {"1": "quick", "2": "named", "3": "ngrok"}.get(choice, "quick")
print(f"Tunnel mode: {TUNNEL_MODE}")


In [ ]:
!pip install comfy-cli
!comfy --skip-prompt install --nvidia

In [ ]:
CUSTOM_NODES = [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/rgthree/rgthree-comfy",
    "https://github.com/yolain/ComfyUI-Easy-Use",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts",
    "https://github.com/Fannovel16/comfyui_controlnet_aux",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/crystian/ComfyUI-Crystools",
    "https://github.com/chflame163/ComfyUI_LayerStyle",
    "https://github.com/kijai/ComfyUI-WanVideoWrapper",
    "https://github.com/ltdrdata/was-node-suite-comfyui",
    "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler",
    "https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch",
    "https://github.com/diodiogod/TTS-Audio-Suite",
    "https://github.com/MohammadAboulEla/ComfyUI-iTools",
    "https://github.com/pixaroma/ComfyUI-Pixaroma",
    #"https://github.com/kijai/ComfyUI-WanAnimatePreprocess",
    #"https://github.com/shiimizu/ComfyUI-TiledDiffusion",
    "https://github.com/kijai/ComfyUI-SCAIL-Pose",
    #"https://github.com/yolain/ComfyUI-Easy-Sam3",
    #"https://github.com/kijai/ComfyUI-MelBandRoFormer",
    #"https://github.com/capitan01R/ComfyUI-Krea2T-Enhancer",
    #"https://github.com/1038lab/ComfyUI-RMBG",
    "https://github.com/cubiq/ComfyUI_essentials"
]

# huggingface: download link (resolve/main/)
# civitai: model page URL or api download URL
# example: https://civitai.com/api/download/models/<modelVersionId>
#          https://civitai.com/api/download/models/<modelVersionId>?type=<Model>&format=<SafeTensor>&fp=<int8> (n variant available)

MODELS ={
    "AUDIO_ENCODERS" : [],	    
    "DETECTION" : [],		 
    "HYPERNETWORKS" : [],		
    "TEXT_ENCODERS" : [
        #"https://huggingface.co/circlestone-labs/Anima/blob/main/split_files/text_encoders/qwen_3_06b_base.safetensors", #Anima
        "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors", #Krea-2
    ],
    "BACKGROUND_REMOVAL" : [],  
    "DIFFUSERS" : [],		 
    "LATENT_UPSCALE_MODELS" : [],	
    "UNET" : [],
    "CHECKPOINTS" : [
        
    ],	     
    "DIFFUSION_MODELS" : [
        #"https://civitai.red/models/2627349/dasiwa-anima", #DaSiWa-ANIMA
        "https://civitai.com/api/download/models/3107017?type=Model&format=SafeTensor&fp=int8", #DaSiWa-Krea2-Turbo-MirroredSkies
    ],	 
    "LORAS" : [],			
    "UPSCALE_MODELS" : [],
    "CLIP" : [],		    
    "EMBEDDINGS" : [],		 
    "MODEL_PATCHES" : [],		
    "VAE" : [
        "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors", #Anima, Krea-2
    ],
    "CLIP_VISION" : [],	    
    "FRAME_INTERPOLATION" : [],  
    "OPTICAL_FLOW" : [],		
    "VAE_APPROX" : [],
    "CONFIGS" : [],		    
    "GEOMETRY_ESTIMATION" : [],  
    "PHOTOMAKER" : [],
    "CONTROLNET" : [],    
    "GLIGEN" : [],		 
    "STYLE_MODELS" : [],
}

MODEL_LINKS ={
    "AUDIO_ENCODERS" : "/root/comfy/ComfyUI/models/audio_encoders/",	    
    "DETECTION" : "/root/comfy/ComfyUI/models/detection/",		 
    "HYPERNETWORKS" : "/root/comfy/ComfyUI/models/hypernetworks/",		
    "TEXT_ENCODERS" : "/root/comfy/ComfyUI/models/text_encoders/",
    "BACKGROUND_REMOVAL" : "/root/comfy/ComfyUI/models/background_removal/",  
    "DIFFUSERS" : "/root/comfy/ComfyUI/models/diffusers/",		 
    "LATENT_UPSCALE_MODELS" : "/root/comfy/ComfyUI/models/latent_upscale_models/",	
    "UNET" : "/root/comfy/ComfyUI/models/unet/",
    "CHECKPOINTS" : "/root/comfy/ComfyUI/models/checkpoints/",	    
    "DIFFUSION_MODELS" : "/root/comfy/ComfyUI/models/diffusion_models/",	 
    "LORAS" : "/root/comfy/ComfyUI/models/loras/",			
    "UPSCALE_MODELS" : "/root/comfy/ComfyUI/models/upscale_models/",
    "CLIP" : "/root/comfy/ComfyUI/models/clip/",		    
    "EMBEDDINGS" : "/root/comfy/ComfyUI/models/embeddings/",		 
    "MODEL_PATCHES" : "/root/comfy/ComfyUI/models/model_patches/",		
    "VAE" : "/root/comfy/ComfyUI/models/vae/",
    "CLIP_VISION" : "/root/comfy/ComfyUI/models/clip_vision/",	    
    "FRAME_INTERPOLATION" : "/root/comfy/ComfyUI/models/frame_interpolation/",  
    "OPTICAL_FLOW" : "/root/comfy/ComfyUI/models/optical_flow/",		
    "VAE_APPROX" : "/root/comfy/ComfyUI/models/vae_approx/",
    "CONFIGS" : "/root/comfy/ComfyUI/models/configs/",		    
    "GEOMETRY_ESTIMATION" : "/root/comfy/ComfyUI/models/geometry_estimation/",  
    "PHOTOMAKER" : "/root/comfy/ComfyUI/models/photomaker/",
    "CONTROLNET" : "/root/comfy/ComfyUI/models/controlnet/",    
    "GLIGEN" : "/root/comfy/ComfyUI/models/gligen/",		 
    "STYLE_MODELS" : "/root/comfy/ComfyUI/models/style_models/",
}


In [ ]:
# ================= MODELS =================
import os, subprocess, pty, select, time, re

BASE = "/root/comfy/ComfyUI/"

_progress_re = re.compile(
    r"\d+%\|.*?it/s|\d+%\|.*? .*?ETA|^\s*$"
)

def run_pty(cmd, env=None):
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(cmd, stdin=slave_fd, stdout=slave_fd, stderr=slave_fd,
                            close_fds=True, env=env)
    os.close(slave_fd)
    buf = ""
    try:
        while True:
            r, _, _ = select.select([master_fd], [], [], 0.2)
            if master_fd in r:
                try:
                    data = os.read(master_fd, 4096)
                except OSError:
                    break
                if not data:
                    break
                buf += data.decode(errors="replace")
                while "\n" in buf:
                    line, buf = buf.split("\n", 1)
                    if not _progress_re.search(line):
                        print(line, flush=True)
            if proc.poll() is not None:
                break
        rest = buf.strip()
        if rest and not _progress_re.search(rest):
            print(rest, end="", flush=True)
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass
    return proc.wait()

try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    HF_TOKEN = client.get_secret("HF_TOKEN")
    CIVITAI_API_KEY = client.get_secret("CIVITAI_API_KEY")
except Exception:
    HF_TOKEN = None
    CIVITAI_API_KEY = None

civitai_flag = []
if CIVITAI_API_KEY:
    civitai_flag = ["--set-civitai-api-token", CIVITAI_API_KEY]

for category, urls in MODELS.items():
    if not urls:
        continue
    dest_dir = MODEL_LINKS[category]
    rel_path = dest_dir.replace(BASE, "").rstrip("/")
    print(f"\n=== {category} ({rel_path}/) ===")

    for url in urls:
        filename = url.rstrip("/").split("/")[-1].split("?")[0]

        if os.path.isfile(os.path.join(dest_dir, filename)):
            print(f"  [SKIP] {filename}")
            continue

        print(f"  [DL]   {filename}")
        cmd = ["comfy", "--skip-prompt", "model", "download", "--url", url,
               "--relative-path", rel_path] + civitai_flag
        env = os.environ.copy()
        if HF_TOKEN:
            env["HF_TOKEN"] = HF_TOKEN

        rc = run_pty(cmd, env=env)
        if rc == 0:
            print(f"  [OK]   {filename}")
        else:
            print(f"  [FAIL] {filename} (exit {rc})")

In [ ]:
# ================= REMOVE MODEL =================
import os

SKIP_EXTS = (".here", ".txt", ".log")

def fmt_size(bytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if bytes < 1024:
            return f"{bytes:.1f} {unit}"
        bytes /= 1024
    return f"{bytes:.1f} TB"

entries = []
idx = 0

for cat, dest_dir in sorted(MODEL_LINKS.items(), key=lambda x: x[0]):
    if not os.path.isdir(dest_dir):
        continue

    files = []
    for f in sorted(os.listdir(dest_dir)):
        fpath = os.path.join(dest_dir, f)
        if not os.path.isfile(fpath):
            continue
        if os.path.splitext(f)[1].lower() in SKIP_EXTS:
            continue
        files.append((f, fpath))

    if not files:
        continue

    rel = dest_dir.replace(BASE, "").rstrip("/")
    print(f"=== {cat} ({rel}/) ===")

    for f, fpath in files:
        idx += 1
        size = fmt_size(os.path.getsize(fpath))
        print(f"  [{idx}] {f} ({size})")
        entries.append((cat, f, fpath))

if not entries:
    print("No models found.")
else:
    inp = input(f"\nNhap so thu tu can xoa (VD: 1,3,5 / 1-3 / all / q): ").strip()

    if inp.lower() == "all":
        selected = entries
    elif inp.lower() == "q" or not inp:
        selected = []
    else:
        nums = set()
        for part in inp.replace(",", " ").split():
            if "-" in part:
                a, b = part.split("-", 1)
                nums.update(range(int(a), int(b) + 1))
            else:
                nums.add(int(part))
        selected = [entries[i - 1] for i in sorted(nums) if 1 <= i <= len(entries)]

    if not selected:
        print("Canceled.")
    else:
        print(f"\nXoa {len(selected)} model(s):")
        for cat, f, fpath in selected:
            print(f"  - {cat}/{f}")
        confirm = input("Xac nhan? (y/N): ").strip().lower()
        if confirm == "y":
            for cat, f, fpath in selected:
                os.remove(fpath)
                print(f"  [OK] Xoa: {cat}/{f}")
        else:
            print("Canceled.")

In [ ]:
# ================= DOWNLOAD CUSTOM NODES =================
import os, time, re, pty, select, subprocess

CUSTOM_NODES_DIR = "/root/comfy/ComfyUI/custom_nodes"

def run_pty_filtered(cmd):
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(cmd, stdin=slave_fd, stdout=slave_fd, stderr=slave_fd, close_fds=True)
    os.close(slave_fd)
    buf = ""
    try:
        while True:
            r, _, _ = select.select([master_fd], [], [], 0.2)
            if master_fd in r:
                try:
                    data = os.read(master_fd, 4096)
                except OSError:
                    break
                if not data:
                    break
                buf += data.decode(errors="replace")
                while "\n" in buf:
                    line, buf = buf.split("\n", 1)
                    if not re.search(r"\d+%\|", line):
                        print(line, flush=True)
        rest = buf.strip()
        if rest and not re.search(r"\d+%\|", rest):
            print(rest, end="", flush=True)
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass
    return proc.wait()

# Get already installed nodes
installed = set()
if os.path.isdir(CUSTOM_NODES_DIR):
    for d in os.listdir(CUSTOM_NODES_DIR):
        d_path = os.path.join(CUSTOM_NODES_DIR, d)
        if os.path.isdir(d_path) and os.path.isfile(os.path.join(d_path, "__init__.py")):
            installed.add(d.lower())

for url in CUSTOM_NODES:
    name = url.rstrip("/").split("/")[-1]
    print(f"\n--- Installing {name} ---")

    if name.lower() in installed:
        print(f"[SKIP] {name} already installed")
        continue

    for attempt in range(2):
        if attempt > 0:
            print(f"[RETRY] {name} (attempt 2)...")
            time.sleep(3)
        rc = run_pty_filtered(["comfy", "--skip-prompt", "node", "install" , url])
        if rc == 0:
            print(f"[OK] {name} installed")
            installed.add(name.lower())
            break
        else:
            print(f"[FAIL] {name} (exit {rc})" + (" - retrying..." if attempt == 0 else ""))

In [ ]:
import os, stat, urllib.request

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
    os.chmod(CLOUDFLARED_PATH, os.stat(CLOUDFLARED_PATH).st_mode | stat.S_IEXEC)
    print("cloudflared ready at", CLOUDFLARED_PATH)
else:
    print("cloudflared already exists at", CLOUDFLARED_PATH)


# Backup & Restore

In [ ]:
import os, re, time, pathlib, subprocess, urllib.request, urllib.parse
import pty, select
from IPython.display import display, HTML

WAIT_SECONDS_SHORT = 20

def run(cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def tail(path, n=3000):
    p = pathlib.Path(path)
    if not p.exists():
        return ""
    return p.read_text(errors="ignore")[-n:]

def run_pty(cmd):
    """Run cmd in PTY and stream output to stdout in realtime. Return exit code."""
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(
        cmd,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        close_fds=True,
    )
    os.close(slave_fd)
    try:
        while True:
            r, _, _ = select.select([master_fd], [], [], 0.2)
            if master_fd in r:
                try:
                    data = os.read(master_fd, 4096)
                except OSError:
                    break
                if not data:
                    break
                text = data.decode(errors="replace")
                print(text, end="", flush=True)
            if proc.poll() is not None:
                break
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass
    return proc.wait()

def find_token_in_log(log_path):
    text = pathlib.Path(log_path).read_text(errors="ignore")
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and "access_token" in line and "refresh_token" in line:
            return line
    return None

def write_rclone_conf(token_json, client_id=None, client_secret=None):
    os.makedirs("/root/.config/rclone", exist_ok=True)
    conf_path = "/root/.config/rclone/rclone.conf"
    with open(conf_path, "w") as f:
        f.write("[" + REMOTE + "]\n")
        f.write("type = drive\n")
        f.write("scope = drive\n")
        if client_id:
            f.write("client_id = " + client_id + "\n")
        if client_secret:
            f.write("client_secret = " + client_secret + "\n")
        f.write("token = " + token_json + "\n")
    os.chmod(conf_path, 0o600)
    return conf_path

run(["apt-get", "update", "-qq"], check=False)
run(["apt-get", "install", "-y", "rclone", "wget", "fuse3", "rsync"], check=False)

token_json = None

gdrive_client_id = None
gdrive_client_secret = None
if USE_CUSTOM_GDRIVE_API and CONFIGURE_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets_client = UserSecretsClient()
        gdrive_client_id = secrets_client.get_secret("GDRIVE_CLIENT_ID")
        gdrive_client_secret = secrets_client.get_secret("GDRIVE_CLIENT_SECRET")
        if gdrive_client_id and gdrive_client_secret:
            print(f"Loaded custom Google API credentials (client_id: ...{gdrive_client_id[-12:]})")
        else:
            print("USE_CUSTOM_GDRIVE_API=True but GDRIVE_CLIENT_ID/SECRET missing in Secrets.")
            print("Falling back to rclone default OAuth app.")
            gdrive_client_id = None
            gdrive_client_secret = None
    except Exception as e:
        print("Cannot load GDRIVE_CLIENT_ID/SECRET:", e)
        print("Falling back to rclone default OAuth app.")
        gdrive_client_id = None
        gdrive_client_secret = None

if CONFIGURE_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
        if secret_token:
            print("Found Kaggle Secret: RCLONE_DRIVE_TOKEN")
            token_json = secret_token

            if USE_CUSTOM_GDRIVE_API and gdrive_client_id:
                write_rclone_conf(token_json, gdrive_client_id, gdrive_client_secret)
                print("Testing token with custom credentials (rclone --dry-run)...")
                test = subprocess.run(
                    ["rclone", "mkdir", REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__stale_test__", "--dry-run"],
                    capture_output=True, text=True, timeout=30,
                )
                err = (test.stderr or test.stdout).lower()
                if test.returncode != 0 and ("invalid_client" in err or "invalid_grant" in err):
                    print("Token was created with rclone default app (incompatible with custom credentials).")
                    print("Triggering re-auth with custom Client ID/Secret...")
                    token_json = None
                else:
                    subprocess.run(["rclone", "purge", REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__stale_test__"],
                                    capture_output=True, timeout=30)
    except Exception as e:
        print("No RCLONE_DRIVE_TOKEN secret found. Need to authenticate.")

    if not token_json:
        print(subprocess.check_output(["rclone", "version"], text=True).splitlines()[0])
        print(subprocess.check_output(["cloudflared", "--version"], text=True).strip())

        subprocess.run(["pkill", "-f", "rclone authorize"], stderr=subprocess.DEVNULL)
        subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)

        rclone_log = "/tmp/rclone_auth.log"
        cloud_log = "/tmp/cloudflared.log"
        open(rclone_log, "w").close()
        open(cloud_log, "w").close()

        print("Starting rclone authorize...")
        auth_env = os.environ.copy()
        if gdrive_client_id:
            auth_env["RCLONE_DRIVE_CLIENT_ID"] = gdrive_client_id
            auth_env["RCLONE_DRIVE_CLIENT_SECRET"] = gdrive_client_secret
            print("Using custom Google API credentials for authorize.")
        subprocess.Popen(
            ["rclone", "authorize", "drive", "--auth-no-open-browser"],
            stdout=open(rclone_log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=auth_env,
        )
        time.sleep(3)

        print("Starting Cloudflare Quick Tunnel for OAuth callback...")
        subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://127.0.0.1:53682"],
            stdout=open(cloud_log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        tunnel_url = None
        local_auth = None
        for _ in range(30):
            cloud_text = tail(cloud_log, 8000)
            rclone_text = tail(rclone_log, 8000)
            tunnels = re.findall(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", cloud_text)
            auths = re.findall(r"http://127\.0\.0\.1:53682[^\s]+", rclone_text)
            if tunnels:
                tunnel_url = tunnels[0]
            if auths:
                local_auth = auths[-1]
            if tunnel_url and local_auth:
                break
            time.sleep(1)

        if not tunnel_url:
            print("=== cloudflared log ===")
            print(tail(cloud_log))
            print("Cannot create tunnel for OAuth.")
        elif not local_auth:
            print("=== rclone log ===")
            print(tail(rclone_log))
            print("Cannot get rclone auth URL.")
        else:
            public_auth = local_auth.replace("http://127.0.0.1:53682", tunnel_url)
            print()
            print("=" * 90)
            print("CLICK LINK NAY DE LOGIN GOOGLE DRIVE:")
            print(public_auth)
            print("=" * 90)
            display(HTML('<a href="' + public_auth + '" target="_blank">\n  Open Google Drive Authorization\n</a>'))

            start = time.time()
            while time.time() - start < WAIT_SECONDS_SHORT:
                token_json = find_token_in_log(rclone_log)
                if token_json:
                    break
                time.sleep(1)

            if not token_json:
                print()
                print("=" * 90)
                print("SAU KHI LOGIN XONG, Google redirect ve http://127.0.0.1:53682/...")
                print("Trinh duyet se bao loi. COPY TOAN BO URL TU THANH DIA CHI")
                print("ROI PASTE XUONG DAY:")
                print("=" * 90)
                pasted = input("Paste URL: ").strip()
                if pasted and "127.0.0.1:53682" in pasted:
                    parsed = urllib.parse.urlparse(pasted)
                    try:
                        urllib.request.urlopen("http://127.0.0.1:53682/?" + parsed.query, timeout=10)
                        print("Callback forwarded. Waiting for token...")
                    except Exception as e:
                        print("Forward error:", e)
                    time.sleep(3)
                    tok = find_token_in_log(rclone_log)
                    if tok:
                        token_json = tok
                        print("Token received!")

            if token_json:
                print("TOKEN RECEIVED.")
                print()
                print("=== LUU LAI TOKEN CHO LAN SAU ===")
                print("Tao Kaggle Secret: Add-ons -> Secrets -> Add a new secret")
                print("  Label: RCLONE_DRIVE_TOKEN")
                print("  Value: (copy dong ben duoi)")
                print()
                print(token_json)
                print()
                print("=" * 60)

if token_json and CONFIGURE_DRIVE:
    write_rclone_conf(token_json, gdrive_client_id, gdrive_client_secret)
    if gdrive_client_id:
        print("rclone.conf written (with custom Google API credentials).")
    else:
        print("rclone.conf written (rclone default OAuth app).")

BACKUP_OK = False
if token_json and CONFIGURE_DRIVE:
    print("\n=== Phase 2: Verify token ===")
    TEST_DIR = REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__rclone_test__"
    verify = subprocess.run(
        ["rclone", "mkdir", TEST_DIR],
        capture_output=True, text=True, timeout=30,
    )
    if verify.returncode != 0:
        print("Token invalid or expired.")
        print("rclone error:", (verify.stderr or verify.stdout).strip())
        print("Re-authenticate: delete RCLONE_DRIVE_TOKEN secret and re-run this cell.")
        BACKUP_OK = False
    else:
        subprocess.run(["rclone", "purge", TEST_DIR], capture_output=True, timeout=30)
        BACKUP_OK = True
        print("Token verified.")
elif not token_json and CONFIGURE_DRIVE:
    print("No token obtained - skipping backup setup.")
else:
    print("Google Drive backup disabled - skipping all phases.")

RESTORE_OK = False
if BACKUP_OK:
    print("\n=== Phase 3: Restore (Drive -> Kaggle) ===")
    lsl = subprocess.run(
        ["rclone", "lsl", REMOTE + ":" + DRIVE_BACKUP_DIR, "--max-depth", "1"],
        capture_output=True, text=True, timeout=30,
    )
    if lsl.returncode != 0:
        RESTORE_OK = True
        print("Cannot list Drive folder (rate limit or access issue).")
        print("Treating as nothing-to-restore. Background sync will retry later.")
    elif not lsl.stdout.strip():
        RESTORE_OK = True
        print("Drive backup folder is empty. Nothing to restore. OK.")
    else:
        n_entries = len(lsl.stdout.strip().splitlines())
        print(f"Found {n_entries} entries on Drive. Auto-restoring (merge, --ignore-times)...")
        os.makedirs(SRC, exist_ok=True)

        RESTORE_OK = True
        for inc in INCLUDES:
            remote_inc = REMOTE + ":" + DRIVE_BACKUP_DIR + "/" + inc
            local_inc = SRC + "/" + inc
            restore_cmd = [
                "rclone", "copy",
                remote_inc,
                local_inc,
                "--exclude", ".__rclone_test__/**",
                "--ignore-times",
                "--stats=2s", "--stats-one-line",
                "--create-empty-src-dirs",
            ]
            for ex in EXCLUDES:
                restore_cmd += ["--exclude", ex]
            print(f"\n--- Restoring {inc} ---")
            for attempt in range(2):
                if attempt > 0:
                    print(f"Retry in 60s (attempt {attempt + 1}/2)...")
                    time.sleep(60)
                exit_code = run_pty(restore_cmd)
                if exit_code == 0:
                    print(f"\nRestore {inc} OK (attempt {attempt + 1}).")
                    break
                else:
                    print(f"\nRestore {inc} attempt {attempt + 1} failed (exit {exit_code}).")
                    if attempt < 1:
                        continue
                    else:
                        print(f"Both attempts failed for {inc}. Skipping.")
                        RESTORE_OK = False
else:
    print("Phase 2 failed - skipping Phase 3.")

INITIAL_BACKUP_OK = False
if BACKUP_OK and RESTORE_OK:
    print("\n=== Phase 4: Initial backup (Kaggle -> Drive) ===")
    src_exists = os.path.exists(SRC)
    src_empty = src_exists and not any(pathlib.Path(SRC).rglob("*"))

    if not src_exists or src_empty:
        print("Source empty (ComfyUI chua co du lieu). Skip initial backup.")
        INITIAL_BACKUP_OK = True
    else:
        all_ok = True
        for inc in INCLUDES:
            local_inc = SRC + "/" + inc
            if not os.path.isdir(local_inc):
                print(f"Skip {inc} (not a directory).")
                continue
            if not any(pathlib.Path(local_inc).rglob("*")):
                print(f"Skip {inc} (empty).")
                continue
            cmd = [
                "rclone", "copy",
                local_inc,
                REMOTE + ":" + DRIVE_BACKUP_DIR + "/" + inc,
                "--stats=2s", "--stats-one-line",
                "--create-empty-src-dirs",
            ]
            for ex in EXCLUDES:
                cmd += ["--exclude", ex]
            print(f"\n--- Backup {inc} ---")
            exit_code = run_pty(cmd)
            if exit_code != 0:
                all_ok = False
                print(f"FAILED: {inc} backup exit code {exit_code}")
        INITIAL_BACKUP_OK = all_ok
        if INITIAL_BACKUP_OK:
            print("\nDONE: initial backup OK")
        else:
            print("\nFAILED: some directories failed. Background sync will NOT start.")
else:
    if not BACKUP_OK:
        print("Phase 2 (verify) failed - skipping Phase 4.")
    elif not RESTORE_OK:
        print("Phase 3 (restore) failed - skipping Phase 4.")

BACKUP_OK = BACKUP_OK and RESTORE_OK and INITIAL_BACKUP_OK

if BACKUP_OK:
    SYNC_SCRIPT = "/kaggle/working/rclone_sync.sh"
    INCLUDES_STR = " ".join(INCLUDES)
    EXCLUDES_STR = " ".join(["--exclude " + ex for ex in EXCLUDES])
    with open(SYNC_SCRIPT, "w") as f:
        f.write("#!/bin/bash\n")
        f.write("set +e\n")
        f.write('INTERVAL=${1:-300}\n')
        f.write('SRC="/root/comfy/ComfyUI"\n')
        f.write('DST="gdrive:ComfyUI"\n')
        f.write('LOG=/kaggle/working/rclone_sync.log\n')
        f.write('INCLUDES="' + INCLUDES_STR + '"\n')
        f.write('EXCLUDES="' + EXCLUDES_STR + '"\n')
        f.write("while true; do\n")
        f.write('  echo "[rclone_copy] $(date) checking $SRC" >> "$LOG"\n')
        f.write('  if [ ! -d "$SRC" ]; then\n')
        f.write('    echo "[rclone_copy] $SRC not found, skip" >> "$LOG"\n')
        f.write('    sleep "$INTERVAL"; continue\n')
        f.write('  fi\n')
        f.write('  for inc in $INCLUDES; do\n')
        f.write('    if [ -z "$(ls -A "$SRC/$inc" 2>/dev/null)" ]; then\n')
        f.write('      echo "[rclone_copy] $SRC/$inc empty, skip" >> "$LOG"\n')
        f.write('      continue\n')
        f.write('    fi\n')
        f.write('    echo "[rclone_copy] $(date) starting copy $inc" >> "$LOG"\n')
        f.write('    rclone copy "$SRC/$inc/" "$DST/$inc" $EXCLUDES --stats=30s --stats-one-line --log-file "$LOG" --log-level INFO 2>> "$LOG"\n')
        f.write('    echo "[rclone_copy] $(date) copy done $inc rc=$?" >> "$LOG"\n')
        f.write('  done\n')
        f.write('  echo "[rclone_copy] $(date) all done, sleeping $INTERVAL" >> "$LOG"\n')
        f.write('  sleep "$INTERVAL"\n')
        f.write("done\n")
    os.chmod(SYNC_SCRIPT, 0o755)
    print("Created " + SYNC_SCRIPT)
    print("Note: using rclone COPY (not sync) - will NOT delete files on Drive.")
else:
    print("rclone_sync.sh NOT created - see errors above.")

print("\nHOAN TAT.")
print(f"BACKUP_OK = {BACKUP_OK}")


# Start ComfyUI

In [ ]:
import os
import subprocess
import time
import urllib.request

# =====================
# STOP OLD PROCESSES
# =====================
for pattern in ["python.*main.py", "comfy", "cloudflared", "ngrok"]:
    subprocess.run(
        ["pkill", "-f", pattern],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

time.sleep(2)

# =====================
# START COMFYUI
# =====================
log_path = "/kaggle/working/comfyui.log"
log_file = open(log_path, "w")

comfyui_process = subprocess.Popen(
    ["comfy", "launch"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)
log_file.close()

print("ComfyUI started (PID:", comfyui_process.pid, ")")
print("Log:", log_path)

time.sleep(15)

# =====================
# HEALTH CHECK
# =====================
try:
    health = urllib.request.urlopen(
        "http://127.0.0.1:8188/system_stats", timeout=10
    ).read().decode()
    print("Health check OK:", health[:200])
except Exception as e:
    print("Health check failed:", e)

# =====================
# BACKGROUND SYNC (only if initial setup succeeded)
# =====================
try:
    BACKUP_OK
except NameError:
    BACKUP_OK = False

if not BACKUP_OK:
    print("=== Initial setup FAILED (see Cell 6 output). Skipping background sync. ===")
elif os.path.exists("/kaggle/working/rclone_sync.sh"):
    sync_proc = subprocess.Popen(
        ["bash", "/kaggle/working/rclone_sync.sh", str(SYNC_INTERVAL)],
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        stdin=subprocess.DEVNULL,
        close_fds=True,
    )
    print(f"Background rclone sync started (PID: {sync_proc.pid})")
    print("Check log: !tail -f /kaggle/working/rclone_sync.log")

    time.sleep(35)
    log_path_sync = "/kaggle/working/rclone_sync.log"
    if os.path.exists(log_path_sync):
        print("--- rclone_sync.log (last 1500 chars) ---")
        print(open(log_path_sync).read()[-1500:])
else:
    print("Sync script not found. Run the Google Drive cell (Cell 6) first.")

# =====================
# TUNNEL
# =====================
if TUNNEL_MODE == "quick":
    print("Starting Cloudflare Quick Tunnel...")
    tunnel_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

elif TUNNEL_MODE == "named":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        token = secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN")
    except Exception as e:
        print("Error getting CLOUDFLARE_TUNNEL_TOKEN:", e)
        print("Set it in Add-ons -> Secrets and re-run.")
        token = None
    if token:
        print("Starting Cloudflare Named Tunnel...")
        tunnel_process = subprocess.Popen(
            ["cloudflared", "tunnel", "run", "--token", token],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
    else:
        print("Cannot start named tunnel without token.")
        tunnel_process = None

elif TUNNEL_MODE == "ngrok":
    import zipfile, shutil, stat
    from kaggle_secrets import UserSecretsClient

    NGROK_PATH = "/kaggle/working/ngrok"
    NGROK_ZIP = "/kaggle/working/ngrok.zip"
    NGROK_URL = "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip"

    if not os.path.exists(NGROK_PATH):
        print("Downloading ngrok...")
        urllib.request.urlretrieve(NGROK_URL, NGROK_ZIP)
        with zipfile.ZipFile(NGROK_ZIP, "r") as z:
            z.extractall("/kaggle/working")
        os.chmod(NGROK_PATH, os.stat(NGROK_PATH).st_mode | stat.S_IEXEC)

    secrets = UserSecretsClient()
    try:
        NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
        NGROK_DOMAIN = secrets.get_secret("NGROK_DOMAIN")
    except Exception as e:
        print("Error getting ngrok secrets:", e)
        print("Set NGROK_AUTHTOKEN and NGROK_DOMAIN in Add-ons -> Secrets.")
        NGROK_AUTHTOKEN = None

    if NGROK_AUTHTOKEN:
        subprocess.run([NGROK_PATH, "config", "add-authtoken", NGROK_AUTHTOKEN], check=False)
        print(f"Starting ngrok... https://{NGROK_DOMAIN}")
        tunnel_process = subprocess.Popen(
            [NGROK_PATH, "http", f"--domain={NGROK_DOMAIN}", "8188"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
    else:
        print("Cannot start ngrok without auth token.")
        tunnel_process = None

# =====================
# KEEP TUNNEL ALIVE
# =====================
if tunnel_process:
    for line in tunnel_process.stdout:
        print(line, end="")
else:
    print("No tunnel process. Local access only: http://127.0.0.1:8188")
    print("Cell will exit immediately. Rerun with a tunnel mode if needed.")
